# `setup-roundtrip` — Prove the Colab and GitHub loop

**Rung R0** · ladder track · part of the *LLM from scratch* curriculum

Work top to bottom. Cells marked `# lfs:solution` are yours to write; please keep the first comment line of every cell as it is. When the last cell has printed a receipt, use **File → Save a copy in GitHub** (same repo, branch `main`, same path, outputs included) and tell Claude it is done.

In [38]:
# lfs:setup bootstrap
# Run this first. It fetches the curriculum repo into the Colab runtime and creates the checker `ck`.
import os, subprocess, sys

LESSON = "setup-roundtrip"
SCALE = "t4"   # "t4" for a free Colab GPU or CPU; change to "up" on a bigger GPU

_REPO = "https://github.com/a3d2ddk/learning-the-magic-of-llms"
_BRANCH = "main"
_ROOT = "/content/llm-from-scratch"

def _git(*args, url=None):
    cmd = ["git", *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).strip()
        if url:
            msg = msg.replace(url, _REPO)   # never echo a token
        raise RuntimeError("git " + args[0] + " failed: " + msg[-400:])
    return r.stdout.strip()

if not os.path.isdir(os.path.join(_ROOT, ".git")):
    _url = _REPO + ".git"
    try:
        from google.colab import userdata
        _tok = userdata.get("GH_TOKEN")
        if _tok:
            _url = _url.replace("https://", "https://x-access-token:" + _tok + "@")
    except Exception:
        pass   # public repo, or not running in Colab
    try:
        _git("clone", "--depth", "1", "-b", _BRANCH, _url, _ROOT, url=_url)
    except RuntimeError as e:
        raise RuntimeError(str(e) + "\nIf the repo is private, add a Colab secret named GH_TOKEN "
                           "(key icon in the sidebar) with read access to it, enable notebook access, and re-run.") from None
else:
    try:
        _git("-C", _ROOT, "pull", "--ff-only")
    except RuntimeError as e:
        print("note: could not update the repo copy:", e)

if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
from course.checks import Checker
ck = Checker(LESSON, scale=SCALE, commit=_git("-C", _ROOT, "rev-parse", "--short", "HEAD"),
             expect=["mean_basic", "mean_single", "softmax_values", "softmax_sums_to_one", "softmax_large_inputs", "softmax_neg_then_zero", "softmax_zero_then_neg", "softmax_near_limit", "softmax_all_negative", "softmax_two_big_ties", "softmax_neg_outlier_first"])
print("ready:", LESSON, "| scale", SCALE, "| repo at", ck.commit)

ready: setup-roundtrip | scale t4 | repo at 0ad24ed


## What this notebook is for

Nothing here is about language models yet. It proves the loop every later lesson depends on:

1. You opened this notebook in Colab from GitHub.
2. The first code cell fetched the curriculum repo into the Colab machine.
3. You write a little code, the checks run, and the last cell prints a **receipt**.
4. You save the notebook back to GitHub **with outputs**, and Claude reads it locally.

If any step misbehaves, that is useful: tell Claude what happened so it gets fixed once, here.

In [39]:
# lfs:setup environment
# What machine did Colab give you? Recorded in the receipt so later lessons can be sized to it.
import platform, shutil, subprocess

print("python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
        print("gpu", gpu, mem_gb, "GB")
        ck.metric("gpu", gpu)
        ck.metric("gpu_mem_gb", mem_gb)
    else:
        ck.metric("gpu", "none")
except ImportError:
    ck.metric("gpu", "torch not installed")
ck.metric("disk_free_gb", round(shutil.disk_usage("/").free / 2**30, 1))

python 3.13.18
torch 2.14.1+cpu | cuda available: False
[metric] gpu = none
[metric] disk_free_gb = 107.6


## Your turn

Cells whose first line is `# lfs:solution` are yours. Keep that first line as it is; it is how
your code is found later.

Write `mean` and `softmax` in plain Python: no imports beyond `math`.

- `mean(xs)`: arithmetic mean of a non-empty list of numbers.
- `softmax(xs)`: list of the same length, each `exp(x_i) / sum_j exp(x_j)`. It must not overflow
  on large inputs such as `[1000.0, 1001.0]`.

In [51]:
# lfs:solution plain_python_warmup
import math

def mean(xs):
    sum= 0.0
    i = 0
    for x in xs:
        i += 1
        sum += x
    return sum / i
    #raise NotImplementedError

def softmax(xs):
    max = xs[0]
    for x in xs:
        if x > max:
            max = x
    norm = 0.0
    for x in xs:
        norm += math.exp(x - max)
    ret = []
    for x in xs:
        ret.append(math.exp(x - max) / norm)
    return ret
    #raise NotImplementedError

In [52]:
# lfs:check mean
# Passing a lambda lets the checker catch an error in your code and report it as a failed check.
ck.close("mean_basic", lambda: mean([1, 2, 3, 4]), 2.5)
ck.close("mean_single", lambda: mean([-7.0]), -7.0)

[PASS] mean_basic — max abs diff 0 (tol 1e-06)
[PASS] mean_single — max abs diff 0 (tol 1e-06)


True

In [53]:
# lfs:check softmax
ck.close("softmax_values", lambda: softmax([0.0, math.log(3.0)]), [0.25, 0.75], tol=1e-12)
ck.close("softmax_sums_to_one", lambda: sum(softmax([0.3, -1.2, 2.0, 0.0])), 1.0, tol=1e-12)
ck.close("softmax_large_inputs", lambda: softmax([1000.0, 1001.0]),
         [1 / (1 + math.e), math.e / (1 + math.e)], tol=1e-12)

[PASS] softmax_values — max abs diff 0 (tol 1e-12)
[PASS] softmax_sums_to_one — max abs diff 0 (tol 1e-12)
[PASS] softmax_large_inputs — max abs diff 0 (tol 1e-12)


True

In [54]:
# lfs:check softmax_stability
# Large magnitudes in both orders, mixed signs, all-negative, and several tied extremes.
# Every exponent should stay <= 0 once you subtract the right value, so none of these should crash.
ck.close("softmax_neg_then_zero", lambda: softmax([-1000.0, 0.0]), [0.0, 1.0], tol=1e-12)
ck.close("softmax_zero_then_neg", lambda: softmax([0.0, -1000.0]), [1.0, 0.0], tol=1e-12)
ck.close("softmax_near_limit", lambda: softmax([0.0, -705.0]), [1.0, math.exp(-705.0)], tol=1e-12)
ck.close("softmax_all_negative", lambda: softmax([-1000.0, -1001.0]),
         [math.e / (1 + math.e), 1 / (1 + math.e)], tol=1e-12)
ck.close("softmax_two_big_ties", lambda: softmax([1000.0, -5.0, 1000.0]), [0.5, 0.0, 0.5], tol=1e-12)
ck.close("softmax_neg_outlier_first", lambda: softmax([-800.0, 3.0, 1.0]),
         [0.0, 1 / (1 + math.exp(-2.0)), math.exp(-2.0) / (1 + math.exp(-2.0))], tol=1e-12)

[PASS] softmax_neg_then_zero — max abs diff 0 (tol 1e-12)
[PASS] softmax_zero_then_neg — max abs diff 0 (tol 1e-12)
[PASS] softmax_near_limit — max abs diff 0 (tol 1e-12)
[PASS] softmax_all_negative — max abs diff 0 (tol 1e-12)
[PASS] softmax_two_big_ties — max abs diff 0 (tol 1e-12)
[PASS] softmax_neg_outlier_first — max abs diff 0 (tol 1e-12)


True

## Save it back

Run the last cell below to print the receipt. Then:

1. **File → Save a copy in GitHub**
2. Choose the curriculum repository and branch `main`
3. Leave the file path exactly as shown, so it overwrites the copy Claude issued
4. Click OK, then tell Claude the notebook is done

If the receipt is missing when Claude reads it, check **Edit → Notebook settings** and make sure
"Omit code cell output when saving this notebook" is switched off.

In [55]:
# lfs:receipt
# Run this last. It prints the receipt Claude reads to verify your work.
_ = ck.receipt()

11/11 checks passed
===LFS-RECEIPT-BEGIN===
{"lesson": "setup-roundtrip", "scale": "t4", "passed": true, "n_passed": 11, "n_checks": 11, "checks": [{"name": "mean_basic", "passed": true, "detail": "max abs diff 0 (tol 1e-06)"}, {"name": "mean_single", "passed": true, "detail": "max abs diff 0 (tol 1e-06)"}, {"name": "softmax_values", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_sums_to_one", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_large_inputs", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_neg_then_zero", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_zero_then_neg", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_near_limit", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_all_negative", "passed": true, "detail": "max abs diff 0 (tol 1e-12)"}, {"name": "softmax_two_big_ties", "passed": true, "detail": "max abs 